# ✈️ Week 5 & 6: Sequential LSTM Model for Turbofan Engine RUL Prediction
### Major Project Deliverables:
- **Week 4 (Member 3):** Sequential sliding-window temporal data preparation grouped by engine ID.
- **Week 5 (Member 4):** Deep Learning LSTM network architecture design, training loop, loss monitoring.
- **Week 6 (Member 3):** Evaluation on unseen test engines (RMSE, MAE, R²), actual vs. predicted RUL visualization.


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# Check execution device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch Version: {torch.__version__} | Active Device: {device}')


## 1. Load NASA C-MAPSS FD001 Dataset
We load training telemetry and test telemetry with ground-truth RUL labels.
The target Remaining Useful Life (RUL) is capped with a piecewise linear ceiling at 125 cycles.


In [ ]:
DATA_DIR = os.path.join('..', 'Dataset') if os.path.exists(os.path.join('..', 'Dataset')) else 'Dataset'

SELECTED_SENSORS = [
    'Cycle',
    'SensorMeasure2', 'SensorMeasure3', 'SensorMeasure4',
    'SensorMeasure7', 'SensorMeasure8', 'SensorMeasure11',
    'SensorMeasure12', 'SensorMeasure13', 'SensorMeasure15',
    'SensorMeasure17', 'SensorMeasure20', 'SensorMeasure21'
]

MAX_RUL_CAP = 125.0

# 1. Load Training Data
cols = ['ID', 'Cycle', 'OpSet1', 'OpSet2', 'OpSet3'] + [f'SensorMeasure{i}' for i in range(1, 22)]
train_df = pd.read_csv(os.path.join(DATA_DIR, 'train_FD001.txt'), sep=r'\s+', header=None).iloc[:, :26]
train_df.columns = cols

# Compute RUL for training engines
max_train_cycles = train_df.groupby('ID')['Cycle'].max().reset_index()
max_train_cycles.columns = ['ID', 'MaxCycle']
train_df = train_df.merge(max_train_cycles, on='ID')
train_df['RUL'] = (train_df['MaxCycle'] - train_df['Cycle']).clip(upper=MAX_RUL_CAP)
train_df.drop('MaxCycle', axis=1, inplace=True)

# 2. Load Test Data & RUL Truth
test_df = pd.read_csv(os.path.join(DATA_DIR, 'test_FD001.txt'), sep=r'\s+', header=None).iloc[:, :26]
test_df.columns = cols

rul_truth = pd.read_csv(os.path.join(DATA_DIR, 'RUL_FD001.txt'), sep=r'\s+', header=None, names=['RUL_truth'])
rul_truth['ID'] = np.arange(1, len(rul_truth) + 1)

max_test_cycles = test_df.groupby('ID')['Cycle'].max().reset_index()
max_test_cycles.columns = ['ID', 'MaxCycle']
test_df = test_df.merge(max_test_cycles, on='ID')
test_df = test_df.merge(rul_truth, on='ID')
test_df['RUL'] = (test_df['MaxCycle'] + test_df['RUL_truth'] - test_df['Cycle']).clip(upper=MAX_RUL_CAP)
test_df.drop(['MaxCycle', 'RUL_truth'], axis=1, inplace=True)

print(f"Train Engines: {train_df['ID'].nunique()} ({len(train_df):,} rows)")
print(f"Test Engines:  {test_df['ID'].nunique()} ({len(test_df):,} rows)")


## 2. Feature Standardization
Standardize sensor readings using `StandardScaler` to bring disparate units (temperatures, static pressures, coolant flows) to zero mean and unit variance.


In [ ]:
scaler = StandardScaler()
train_df[SELECTED_SENSORS] = scaler.fit_transform(train_df[SELECTED_SENSORS])
test_df[SELECTED_SENSORS]  = scaler.transform(test_df[SELECTED_SENSORS])

print('Standardization complete. Feature preview:')
train_df[SELECTED_SENSORS].head()


## 3. Sliding Time-Window Sequence Generation (Week 4 Task)
LSTMs require sequential 3D input tensors: `(num_samples, window_size, num_features)`.
- We use a sliding window of **$W = 30$ flight cycles**.
- Sequences are generated strictly **within each engine**, preventing temporal leakage across engines.


In [ ]:
SEQUENCE_LENGTH = 30

def gen_train_sequences(df, seq_len, feature_cols):
    sequences, labels = [], []
    for engine_id in df['ID'].unique():
        engine_data = df[df['ID'] == engine_id]
        if len(engine_data) < seq_len:
            continue
        feat_matrix = engine_data[feature_cols].values
        rul_array   = engine_data['RUL'].values
        for i in range(len(engine_data) - seq_len + 1):
            sequences.append(feat_matrix[i:i + seq_len])
            labels.append(rul_array[i + seq_len - 1])
    return np.array(sequences, dtype=np.float32), np.array(labels, dtype=np.float32)

def gen_test_sequences(df, seq_len, feature_cols):
    sequences, labels = [], []
    for engine_id in df['ID'].unique():
        engine_data = df[df['ID'] == engine_id]
        feat_matrix = engine_data[feature_cols].values
        rul_array   = engine_data['RUL'].values
        if len(engine_data) >= seq_len:
            sequences.append(feat_matrix[-seq_len:])
            labels.append(rul_array[-1])
        else:
            pad_len = seq_len - len(engine_data)
            padding = np.repeat(feat_matrix[0:1], pad_len, axis=0)
            padded  = np.vstack([padding, feat_matrix])
            sequences.append(padded)
            labels.append(rul_array[-1])
    return np.array(sequences, dtype=np.float32), np.array(labels, dtype=np.float32)

X_train, y_train = gen_train_sequences(train_df, SEQUENCE_LENGTH, SELECTED_SENSORS)
X_test,  y_test  = gen_test_sequences(test_df, SEQUENCE_LENGTH, SELECTED_SENSORS)

print(f'Training Sequences Shape: {X_train.shape} (Samples, Window, Channels)')
print(f'Test Sequences Shape:     {X_test.shape}')


## 4. LSTM Neural Network Architecture (Week 5 Task)
Architecture:
- **Input:** `(batch_size, 30, 13)`
- **LSTM Layer 1:** 64 units, batch_first=True
- **Dropout:** 0.2
- **LSTM Layer 2:** 32 units, batch_first=True
- **Fully Connected 1:** 32 -> 16 + ReLU
- **Output Layer:** 16 -> 1 (Continuous predicted RUL)


In [ ]:
class TurbofanLSTM(nn.Module):
    def __init__(self, input_dim=13, hidden_dim1=64, hidden_dim2=32, dropout=0.2):
        super().__init__()
        self.lstm1 = nn.LSTM(input_dim, hidden_dim1, batch_first=True)
        self.dropout1 = nn.Dropout(dropout)
        self.lstm2 = nn.LSTM(hidden_dim1, hidden_dim2, batch_first=True)
        self.fc1 = nn.Linear(hidden_dim2, 16)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(16, 1)

    def forward(self, x):
        out, _ = self.lstm1(x)
        out = self.dropout1(out)
        out, _ = self.lstm2(out)
        out = out[:, -1, :]  # Final time-step feature representation
        out = self.relu(self.fc1(out))
        return self.fc2(out).squeeze(-1)

model = TurbofanLSTM(input_dim=len(SELECTED_SENSORS)).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
print(model)


## 5. Model Training & Loss Monitoring (Week 5 Task)
We split training sequences into 80% train and 20% validation sets.
We monitor RMSE loss across 25 epochs.


In [ ]:
BATCH_SIZE = 64
EPOCHS = 25

split_idx = int(len(X_train) * 0.8)
train_dataset = TensorDataset(torch.tensor(X_train[:split_idx]), torch.tensor(y_train[:split_idx]))
val_dataset   = TensorDataset(torch.tensor(X_train[split_idx:]), torch.tensor(y_train[split_idx:]))

train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)

train_losses, val_losses = [], []
best_val_loss = float('inf')

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_train_loss = 0.0
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        loss = criterion(model(bx), by)
        loss.backward()
        optimizer.step()
        running_train_loss += loss.item() * len(by)
        
    epoch_train_rmse = np.sqrt(running_train_loss / len(train_dataset))
    
    model.eval()
    running_val_loss = 0.0
    with torch.no_grad():
        for vx, vy in val_loader:
            vx, vy = vx.to(device), vy.to(device)
            v_loss = criterion(model(vx), vy)
            running_val_loss += v_loss.item() * len(vy)
            
    epoch_val_rmse = np.sqrt(running_val_loss / len(val_dataset))
    train_losses.append(epoch_train_rmse)
    val_losses.append(epoch_val_rmse)
    
    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch [{epoch:2d}/{EPOCHS}] - Train RMSE: {epoch_train_rmse:6.2f} | Val RMSE: {epoch_val_rmse:6.2f}')


In [ ]:
# Plot Training & Validation Loss Curve (Week 5 Deliverable)
plt.figure(figsize=(9, 4.5))
plt.plot(range(1, EPOCHS + 1), train_losses, label='Train Loss (RMSE)', color='#0ea5e9', lw=2)
plt.plot(range(1, EPOCHS + 1), val_losses,   label='Validation Loss (RMSE)', color='#f59e0b', lw=2, linestyle='--')
plt.title('Week 5 Deliverable: LSTM Training vs. Validation Loss Curves', fontsize=12, fontweight='bold')
plt.xlabel('Epochs')
plt.ylabel('Loss (RMSE Cycles)')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


## 6. Evaluation on Unseen Test Engines (Week 6 Task)
We evaluate the trained LSTM on the test set (`100 test engines`) to predict their final RUL.
Metrics: **RMSE, MAE, R²**.


In [ ]:
model.eval()
with torch.no_grad():
    test_tensor = torch.tensor(X_test).to(device)
    raw_preds   = model(test_tensor).cpu().numpy()
    test_preds  = np.clip(raw_preds, 0.0, MAX_RUL_CAP)

test_rmse = np.sqrt(mean_squared_error(y_test, test_preds))
test_mae  = mean_absolute_error(y_test, test_preds)
test_r2   = r2_score(y_test, test_preds)

print('====================================================')
print('       LSTM MODEL TEST PERFORMANCE METRICS          ')
print('====================================================')
print(f'Test RMSE : {test_rmse:.2f} cycles')
print(f'Test MAE  : {test_mae:.2f} cycles')
print(f'Test R^2  : {test_r2:.4f}')
print('====================================================')


In [ ]:
# Actual vs. Predicted RUL Curve for all 100 test engines (Week 6 Deliverable)
plt.figure(figsize=(12, 5))
engine_idx = np.arange(1, len(y_test) + 1)
plt.plot(engine_idx, y_test, label='Actual RUL (Ground Truth)', color='#10b981', lw=2)
plt.plot(engine_idx, test_preds, label='LSTM Predicted RUL', color='#ef4444', lw=1.8, linestyle='--')
plt.title(f'Week 6 Deliverable: Actual vs. Predicted RUL Across 100 Test Engines (RMSE: {test_rmse:.2f})', fontsize=12, fontweight='bold')
plt.xlabel('Engine ID')
plt.ylabel('Remaining Useful Life (Cycles)')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


## 7. Model Performance Comparison
Comparison of classical ML baseline vs. deep learning sequential models on NASA C-MAPSS (FD001 official test set, 100 engines):
| Model Architecture | Test RMSE (Cycles) | Test MAE (Cycles) | Test R² |
|---|---|---|---|
| Linear Regression (Baseline) | 22.19 | 17.37 | 0.69 |
| Random Forest (Ensemble) | 19.76 | 14.95 | 0.76 |
| **XGBoost (Tuned Regressor)** | **18.91** | **14.35** | **0.78** |
| **LSTM Network (Sequential DL)** | **13.23** | **9.45** | **0.89** |
